# SpectralConv2d - the heart of the FNO

*Walkthrough notebook for `neural_operators/layers/spectral_conv.py`.*

A neural operator layer has to apply an **integral operator** to a function $v$:

$$(\mathcal K v)(x)=\int_D \kappa(x,y)\,v(y)\,dy .$$

Doing that directly on $J$ grid points costs $O(J^2)$. The FNO's trick:

1. If the kernel is **translation invariant**, $\kappa(x,y)=\kappa(x-y)$, the integral is a *convolution*.
2. By the **convolution theorem**, a convolution becomes an element-wise product in Fourier space:
   $\mathcal F[\mathcal K v]=\mathcal F[\kappa]\cdot\mathcal F[v]$.
3. So we skip $\kappa$ entirely and **learn its Fourier coefficients** $R_k$ for the lowest $k_{max}$ modes only.

The layer is therefore: `FFT -> multiply low modes by learned complex weights -> zero the rest -> inverse FFT`,
costing $O(J\log J)$. Keeping only low modes means the parameter count does *not* depend on the grid size,
which is why an FNO trained on one resolution can be run on another.

## Setup

The first cell makes the repository importable when this notebook is opened from `notebooks/source/`; the second holds the module's imports.

In [ ]:
# Make the repo root importable so `import neural_operators` works from notebooks/source/.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "neural_operators").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [ ]:
import torch
import torch.nn as nn

## The layer

Things to notice in the code below:

- **Two weight tensors** (`weight1`, `weight2`). `rfft2` stores only non-negative frequencies along the *last*
  axis, but along the *first* axis it keeps both positive and negative frequencies. The low positive block and
  the low negative block (which wraps around to the end of the axis) each need their own weights.
- Weights are **complex** (`torch.cfloat`) because Fourier coefficients are complex numbers.
- `_complex_mul2d` is an `einsum`: for every retained mode $(x,y)$ it applies a matrix that maps
  `in_channels -> out_channels`.
- `forward` never touches modes above `modes1`/`modes2`; they stay zero, which acts as a low-pass filter.

In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1  # number of retained Fourier modes, first spatial dim
        self.modes2 = modes2  # number of retained Fourier modes, second spatial dim

        scale = 1.0 / (in_channels * out_channels)
        # Two separate weight blocks are needed because rfft2's output only stores the
        # non-negative frequencies along the last axis; the corresponding negative
        # frequencies along the *other* axis still need their own independent weights
        # (real signals have conjugate-symmetric spectra, but that symmetry links a
        # positive-k1 mode to a negative-k1 mode, not to itself).
        self.weight1 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat)
        )
        self.weight2 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat)
        )

    @staticmethod
    def _complex_mul2d(x, weight):
        # x:      (batch, in_channels,  modes1, modes2)
        # weight: (in_channels, out_channels, modes1, modes2)
        return torch.einsum("bixy,ioxy->boxy", x, weight)

    def forward(self, x):
        # x: (batch, in_channels, size1, size2)
        batch_size = x.shape[0]
        size1, size2 = x.shape[-2], x.shape[-1]

        x_ft = torch.fft.rfft2(x)  # (batch, in_channels, size1, size2//2 + 1), complex

        out_ft = torch.zeros(
            batch_size, self.out_channels, size1, size2 // 2 + 1,
            dtype=torch.cfloat, device=x.device,
        )
        # Low positive frequencies (top-left block of the truncated spectrum).
        out_ft[:, :, : self.modes1, : self.modes2] = self._complex_mul2d(
            x_ft[:, :, : self.modes1, : self.modes2], self.weight1
        )
        # Low negative frequencies along dim -2 (wrap-around; bottom-left block).
        out_ft[:, :, -self.modes1 :, : self.modes2] = self._complex_mul2d(
            x_ft[:, :, -self.modes1 :, : self.modes2], self.weight2
        )

        x = torch.fft.irfft2(out_ft, s=(size1, size2))
        return x

## Try it

Apply the layer to random data and check the shapes. Then verify the *discretization-invariance* claim:
the **same weights** accept a 32x32 input and a 64x64 input, and the number of parameters never changes.

In [ ]:
import torch

layer = SpectralConv2d(in_channels=4, out_channels=8, modes1=6, modes2=6)
x = torch.randn(2, 4, 32, 32)             # (batch, channels, H, W)
y = layer(x)
print("input :", tuple(x.shape))
print("output:", tuple(y.shape))
print("parameters:", sum(p.numel() for p in layer.parameters()), "(complex numbers count as 1 each)")

In [ ]:
# Same weights, different resolution
for n in (16, 32, 64, 128):
    out = layer(torch.randn(1, 4, n, n))
    print(f"{n:>3}x{n:<3} -> {tuple(out.shape)}")

In [ ]:
# Low-pass behaviour: a very high-frequency input is almost completely removed.
import math
n = 64
grid = torch.linspace(0, 1, n + 1)[:-1]
X, Y = torch.meshgrid(grid, grid, indexing="ij")
low  = torch.sin(2 * math.pi * 2  * X)    # frequency 2  (kept, since < 6 modes)
high = torch.sin(2 * math.pi * 20 * X)    # frequency 20 (dropped)
probe = SpectralConv2d(1, 1, modes1=6, modes2=6)
print("energy out | low-frequency input :", probe(low[None, None]).norm().item())
print("energy out | high-frequency input:", probe(high[None, None]).norm().item())